# 04 — Merge and analyze

Merge NBA 2025-26 odds + Polymarket + box scores, then test which market
(prices: Polymarket vs sportsbook) better prices in refereeing, star absences,
and individual performance (recent form).

Outputs:
  - nba_merged_analysis.csv : one row per game with all sources + derived fields
  - analysis_results.txt    : headline test results


In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
# project root: works whether this runs from notebooks/ or the project root
def _find_proj(start):
    p = Path(start).resolve()
    for c in [p, *p.parents]:
        if (c / 'data' / 'raw' / 'nba_2025_26_odds_scores.csv').exists():
            return c
        if (c / 'data' / 'raw').is_dir() and (c / 'notebooks').is_dir():
            return c
    return p
PROJ = _find_proj(Path.cwd())
RAW = PROJ / 'data' / 'raw'
INTERIM = PROJ / 'data' / 'interim'
OUT = PROJ / 'outputs'
RES = PROJ / 'results'
CACHE = PROJ / 'cache'
base = pd.read_csv(RAW / 'nba_2025_26_odds_scores.csv')
poly = pd.read_csv(RAW / 'polymarket_odds.csv')
team = pd.read_csv(RAW / 'boxscores_team.csv')
play = pd.read_csv(RAW / 'boxscores_player.csv')

KEY = ['date', 'away_team', 'home_team']


In [ ]:
# ---------- base merge: odds + polymarket ----------
df = base.merge(poly, on=KEY, suffixes=('', '_poly'))
df = df.drop(columns=[c for c in df.columns if c.startswith('notes')])
df['margin'] = df['home_score'] - df['away_score']
df['home_win'] = (df['margin'] > 0).astype(int)
df['actual_total'] = df['home_score'] + df['away_score']
df['over_hit'] = (df['actual_total'] > df['total']).astype(int)
df['home_cover'] = ((df['margin'] + df['home_spread']) > 0).astype(int)


def imp(ml):
    return 100 / (ml + 100) if ml > 0 else -ml / (-ml + 100)


df['sb_home_imp'] = df['home_ml'].apply(imp)
df['sb_away_imp'] = df['away_ml'].apply(imp)
df['sb_home_prob'] = df['sb_home_imp'] / (df['sb_home_imp'] + df['sb_away_imp'])
df['sb_spread_margin'] = -df['home_spread']


In [ ]:
# ---------- team box scores pivoted to home/away ----------
th = team[team['team'] == team['home_team']].set_index(KEY).add_prefix('home_').drop(columns=['home_team'])
ta = team[team['team'] == team['away_team']].set_index(KEY).add_prefix('away_').drop(columns=['away_team'])
df = df.set_index(KEY).join(th).join(ta).reset_index()
df['officials'] = df['home_officials']
df['total_pf'] = df['home_pf'] + df['away_pf']
df['total_fta'] = df['home_fta'] + df['away_fta']
df['total_tov'] = df['home_tov'] + df['away_tov']


In [ ]:
# ---------- crew foul tendency, leave-one-game-out ----------
g = df[['date', 'away_team', 'home_team', 'total_pf', 'officials']].copy()
ref_games = {}
for i, r in g.iterrows():
    for ref in str(r['officials']).split('; '):
        ref_games.setdefault(ref, []).append((i, r['total_pf']))
loo = {}
for ref, lst in ref_games.items():
    tot, n = sum(v for _, v in lst), len(lst)
    for i, v in lst:
        loo[(ref, i)] = (tot - v) / (n - 1) if n > 1 else np.nan
df['crew_foul_loo'] = [
    np.nanmean([loo.get((ref, i), np.nan) for ref in str(r['officials']).split('; ')])
    for i, r in g.iterrows()
]


In [ ]:
# ---------- player aggregates per team-game ----------
play['played'] = play['mp'] != 'DNP'
season = play[play['played']].groupby(['team', 'player'])['pts'].agg(['sum', 'count'])
season['ppg'] = season['sum'] / season['count']
top3, top1 = {}, {}
for tm, sub in season.groupby(level=0):
    ranked = sub.sort_values('ppg', ascending=False).index.get_level_values(1)
    top3[tm] = list(ranked[:3])
    top1[tm] = ranked[0]

# trailing 5-game PPG (pregame info only: prior games, played only)
play_sorted = play[play['played']].sort_values(['team', 'player', 'date'])
play_sorted = play_sorted.copy()
play_sorted['trail5_ppg'] = (
    play_sorted.groupby(['team', 'player'])['pts']
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
)
trail = {(r['date'], r['team'], r['player']): r['trail5_ppg']
         for _, r in play_sorted.iterrows()}

rows = {}
for (date, aw, hm, tm), sub in play.groupby(KEY + ['team']):
    dnp_top3 = any(
        ((sub['player'] == p) & (sub['mp'] == 'DNP')).any() for p in top3[tm]
    )
    played = sub[sub['played']]
    top_pts = played['pts'].max() if len(played) else np.nan
    top3_pts = played[played['player'].isin(top3[tm])]['pts'].sum()
    top1_trail = trail.get((date, tm, top1[tm]), np.nan)
    rows[(date, aw, hm, tm)] = (dnp_top3, top_pts, top3_pts, top1_trail)

recs = []
for _, r in df.iterrows():
    k = (r['date'], r['away_team'], r['home_team'])
    hd = rows.get(k + (r['home_team'],), (False, np.nan, np.nan, np.nan))
    ad = rows.get(k + (r['away_team'],), (False, np.nan, np.nan, np.nan))
    recs.append({
        'home_star_dnp': hd[0], 'away_star_dnp': ad[0],
        'home_top_pts': hd[1], 'away_top_pts': ad[1],
        'home_top3_pts': hd[2], 'away_top3_pts': ad[2],
        'home_top1_trail5': hd[3], 'away_top1_trail5': ad[3],
    })
pa = pd.DataFrame(recs)
df = pd.concat([df.reset_index(drop=True), pa], axis=1)
df['any_star_dnp'] = df['home_star_dnp'] | df['away_star_dnp']
df['form_diff'] = df['home_top1_trail5'] - df['away_top1_trail5']

df.to_csv(OUT / 'nba_merged_analysis.csv', index=False)
print('merged rows:', len(df), '| cols:', len(df.columns))


In [ ]:
# ---------- helpers ----------
def brier(p, y):
    m = ~np.isnan(p)
    return float(np.mean((p[m] - y[m]) ** 2)), int(m.sum())


def logloss(p, y):
    m = ~np.isnan(p)
    pc = np.clip(p[m], 1e-6, 1 - 1e-6)
    return float(-np.mean(y[m] * np.log(pc) + (1 - y[m]) * np.log(1 - pc))), int(m.sum())


def ols(X, y, names):
    X = np.column_stack([np.ones(len(np.asarray(X[0])))] + [np.asarray(c, float) for c in X])
    mask = ~np.isnan(X).any(axis=1) & ~np.isnan(y)
    X, y = X[mask], np.asarray(y, float)[mask]
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    dof = len(y) - X.shape[1]
    s2 = (resid @ resid) / dof
    cov = s2 * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    return {n: (round(b, 4), round(b / s, 2)) for n, b, s in
            zip(['const'] + names, beta, se)}, int(mask.sum())


out = []
out.append('=== HEAD-TO-HEAD CALIBRATION (home win) ===')
bp, npoly = brier(df['poly_home_prob'].values, df['home_win'].values)
bs, _ = brier(df['sb_home_prob'].values, df['home_win'].values)
lp, _ = logloss(df['poly_home_prob'].values, df['home_win'].values)
ls, _ = logloss(df['sb_home_prob'].values, df['home_win'].values)
out.append(f'Polymarket Brier {bp:.4f} (n={npoly}) | Sportsbook Brier {bs:.4f}')
out.append(f'Polymarket log loss {lp:.4f} | Sportsbook log loss {ls:.4f}')
d = df.dropna(subset=['poly_home_prob']).copy()
d['poly_err'] = (d['poly_home_prob'] - d['home_win']).abs()
d['sb_err'] = (d['sb_home_prob'] - d['home_win']).abs()
out.append(f"Games where Poly was closer to the outcome: {(d['poly_err'] < d['sb_err']).mean()*100:.1f}% "
           f"(SB closer {(d['sb_err'] < d['poly_err']).mean()*100:.1f}%, tie {(d['poly_err']==d['sb_err']).mean()*100:.1f}%)")
out.append(f"Mean |error| Poly {d['poly_err'].mean():.3f} vs SB {d['sb_err'].mean():.3f}")


In [ ]:
out.append('\n=== REFEREEING: crew foul tendency (leave-one-out) ===')
out.append(f"Mean total fouls/game: {df['total_pf'].mean():.2f}; crew LOO foul rate range "
           f"{df['crew_foul_loo'].min():.1f}–{df['crew_foul_loo'].max():.1f}")
out.append(f"corr(crew foul rate, actual total) = {df['crew_foul_loo'].corr(df['actual_total']):.3f}")
out.append(f"corr(crew foul rate, sportsbook total line) = {df['crew_foul_loo'].corr(df['total']):.3f}")
out.append(f"corr(crew foul rate, Poly total line) = {df['crew_foul_loo'].corr(df['poly_total_line']):.3f}")
out.append(f"corr(crew foul rate, Poly over prob) = {df['crew_foul_loo'].corr(df['poly_over_prob']):.3f}")
out.append(f"corr(crew foul rate, actual total MINUS sportsbook total) = "
           f"{df['crew_foul_loo'].corr(df['actual_total'] - df['total']):.3f}  <- do lines miss foul crews?")
out.append(f"corr(crew foul rate, Poly over prob MINUS 0.5) = "
           f"{df['crew_foul_loo'].corr(df['poly_over_prob'] - 0.5):.3f}")
df['foul_tercile'] = pd.qcut(df['crew_foul_loo'], 3, labels=['low', 'mid', 'high'])
t = df.groupby('foul_tercile', observed=True).agg(
    n=('home_win', 'size'), over_hit=('over_hit', 'mean'),
    sb_total=('total', 'mean'), poly_over=('poly_over_prob', 'mean'),
    actual_total=('actual_total', 'mean')).round(3)
out.append('By crew foul tercile:\n' + t.to_string())
r, n = ols([df['total'], df['crew_foul_loo']], df['actual_total'],
           ['sb_total', 'crew_foul'])
out.append(f'OLS actual_total ~ sb_total + crew_foul (n={n}): {r}')
r, n = ols([df['poly_total_line'], df['crew_foul_loo']], df['poly_over_prob'],
           ['poly_total_line', 'crew_foul'])
out.append(f'OLS poly_over_prob ~ poly_total_line + crew_foul (n={n}): {r}')


In [ ]:
out.append('\n=== STAR ABSENCES (top-3 scorer by season PPG DNP) ===')
out.append(f"Games with a top-3 scorer DNP: {df['any_star_dnp'].sum()} "
           f"(home {df['home_star_dnp'].sum()}, away {df['away_star_dnp'].sum()})")
for label, sub in [('DNP games', df[df['any_star_dnp']]),
                   ('no-DNP games', df[~df['any_star_dnp']])]:
    bp, n1 = brier(sub['poly_home_prob'].values, sub['home_win'].values)
    bs, _ = brier(sub['sb_home_prob'].values, sub['home_win'].values)
    out.append(f'{label} (n={len(sub)}): Brier Poly {bp:.4f} vs SB {bs:.4f}')
# does the market under-react to the absence? team missing a star: win rate vs predicted
for side in ['home', 'away']:
    sub = df[df[f'{side}_star_dnp']]
    if side == 'home':
        pred_poly, pred_sb, actual = sub['poly_home_prob'], sub['sb_home_prob'], sub['home_win']
    else:
        pred_poly, pred_sb, actual = 1 - sub['poly_home_prob'], 1 - sub['sb_home_prob'], 1 - sub['home_win']
    out.append(f'{side} star DNP (n={len(sub)}): {side} team won {actual.mean()*100:.1f}% of games; '
               f'Poly implied {pred_poly.mean()*100:.1f}%, SB implied {pred_sb.mean()*100:.1f}%')


In [ ]:
out.append('\n=== INDIVIDUAL PERFORMANCE: top scorer recent form (trailing 5-game PPG, pregame) ===')
both = df[df['home_top1_trail5'].notna() & df['away_top1_trail5'].notna()
          & ~df['home_star_dnp'] & ~df['away_star_dnp']].copy()
out.append(f'Games with form for both top scorers (both played): {len(both)}')
out.append(f"corr(form_diff, SB home prob) = {both['form_diff'].corr(both['sb_home_prob']):.3f}")
out.append(f"corr(form_diff, Poly home prob) = {both['form_diff'].corr(both['poly_home_prob']):.3f}")
r, n = ols([both['sb_home_prob'], both['form_diff']], both['home_win'],
           ['sb_prob', 'form_diff'])
out.append(f'OLS home_win ~ sb_prob + form_diff (n={n}): {r}')
r, n = ols([both['poly_home_prob'], both['form_diff']], both['home_win'],
           ['poly_prob', 'form_diff'])
out.append(f'OLS home_win ~ poly_prob + form_diff (n={n}): {r}')
# realized star performance surprise
both['home_top_surprise'] = both['home_top_pts'] - both['home_top1_trail5']
both['away_top_surprise'] = both['away_top_pts'] - both['away_top1_trail5']
both['surprise_diff'] = both['home_top_surprise'] - both['away_top_surprise']
out.append(f"corr(star surprise diff, margin) = {both['surprise_diff'].corr(both['margin']):.3f}")
r, n = ols([both['sb_home_prob'], both['surprise_diff']], both['home_win'],
           ['sb_prob', 'surprise_diff'])
out.append(f'OLS home_win ~ sb_prob + star surprise (n={n}): {r}  <- realized performance moves outcomes beyond the line')

txt = '\n'.join(out)
print(txt)
open(RES / 'analysis_results.txt', 'w').write(txt + '\n')
